# Traffic-light CNN on the Tensil accelerator (PYNQ board)
Upload this notebook to the board's Jupyter file browser (root = `/home/xilinx/jupyter_notebooks`). Expected files there:

- `tl_tcu.bit` and `tl_tcu.hwh` (same base name)
- `tcu_pynq/` (Tensil's Python driver folder)
- `tl/` containing `tl_cnn_onnx_pynqz1.tmodel`, `.tdata`, `.tprog`, `test_images_uint8_NHWC.npy`, `test_labels.npy`, `class_names.json`

In [ ]:
# CELL 1: load the accelerator bitstream and the Tensil driver
import sys, os, time, json
BASE = '/home/xilinx/jupyter_notebooks'
sys.path.append(BASE)
import numpy as np
import pynq
from pynq import Overlay
print('PYNQ version:', pynq.__version__)
from tcu_pynq.driver import Driver
from tcu_pynq.architecture import pynqz1

overlay = Overlay(BASE + '/tl_tcu.bit')
tcu = Driver(pynqz1, overlay.axi_dma_0)       # the DMA in the Vivado design must be called axi_dma_0
print('accelerator loaded; array size', tcu.arch.array_size)
from pynq import Clocks
print('PL clock (MHz):', Clocks.fclk0_mhz)   # should match the frequency set in Vivado (50)

In [ ]:
# CELL 2: load the compiled model and show its input/output names
D = BASE + '/tl/'
tcu.load_model(D + 'tl_cnn_onnx_pynqz1.tmodel')
IN_NAME  = tcu.model.inputs[0].name
OUT_NAME = tcu.model.outputs[0].name
print('model input name :', IN_NAME)
print('model output name:', OUT_NAME)

X   = np.load(D + 'test_images_uint8_NHWC.npy')     # N x 32 x 32 x 3, uint8
y   = np.load(D + 'test_labels.npy')
CLS = json.load(open(D + 'class_names.json'))       # ['green', 'red', 'yellow']
print(len(X), 'test images, classes:', CLS)

def prep(img_u8):
    img = img_u8.astype(np.float32) / 255.0                                  # 0..1, same as training
    img = np.pad(img, [(0, 0), (0, 0), (0, tcu.arch.array_size - 3)])        # channels 3 -> 8
    return img.reshape((-1, tcu.arch.array_size))                            # 1024 x 8

def classify(img_u8):
    return tcu.run({IN_NAME: prep(img_u8)})[OUT_NAME][:3]

In [ ]:
# CELL 3: one image
i = 0
t0 = time.time(); scores = classify(X[i]); dt = time.time() - t0
print('scores (green, red, yellow):', np.round(scores, 3))
print('predicted:', CLS[int(np.argmax(scores))], '| true:', CLS[y[i]], '| time: %.2f ms' % (dt * 1000))

In [ ]:
# CELL 4: whole test set: accuracy and time per image ON THE FPGA
cm = np.zeros((3, 3), dtype=int)
t0 = time.time()
for i in range(len(X)):
    cm[y[i], int(np.argmax(classify(X[i])))] += 1
dt = (time.time() - t0) / len(X)
acc = np.trace(cm) / cm.sum()
print(f'FPGA accuracy: {np.trace(cm)} / {cm.sum()} = {acc:.4f}')
print(f'time per image: {dt*1000:.2f} ms (includes Python and DMA overhead, not just the accelerator)')
print('\nconfusion matrix (rows = true, cols = predicted)', CLS)
print(cm)